# 📄 arXiv 論文爬蟲 — 物件導向版本
### 領域：電腦視覺 / 圖形辨識（Computer Vision & Image Recognition）

---

## 📌 這份 Notebook 的學習路徑

```
階段一：認識 arXiv API 網址長什麼樣
    ↓
階段二：用 Python 發送請求，看看伺服器回傳什麼
    ↓
階段三：認識 BeautifulSoup，了解它怎麼解析 XML
    ↓
階段四：用 BeautifulSoup 逐步取出每個欄位
    ↓
階段五：用物件導向（Class）整理程式碼
    ↓
階段六：執行爬蟲，搜尋 CV 論文，儲存結果
```

---

## 📌 函式版本 vs 物件導向版本的差別

| | 函式版本 | 物件導向版本 |
|---|---|---|
| 組織方式 | 一個獨立函式 | 把資料和方法包在 Class 裡 |
| 狀態保存 | 每次呼叫都要重新傳參數 | 資料存在物件裡，隨時可以取用 |
| 擴充性 | 需要新增更多函式 | 直接在 Class 裡新增方法 |
| 適合場景 | 簡單單次任務 | 需要管理狀態、多次操作 |

---
# 階段一：認識 arXiv API 網址

在寫任何程式之前，先用**瀏覽器**直接打開下面這個網址，看看 API 回傳什麼：

👉 http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2

---

### 網址的組成結構

```
http://export.arxiv.org/api/query
        ↑ 這是 API 的基本網址（Base URL）

?search_query=cat:cs.CV&max_results=2
 ↑ 問號後面是參數，用 & 分隔多個參數
```

| 參數 | 意思 | 範例值 |
|------|------|--------|
| `search_query` | 搜尋什麼 | `cat:cs.CV`（電腦視覺分類） |
| `max_results` | 要幾筆資料 | `10` |
| `start` | 從第幾筆開始 | `0`（第一頁） |
| `sortBy` | 排序方式 | `submittedDate` |
| `sortOrder` | 升冪或降冪 | `descending`（最新優先） |

---
# 階段二：用 Python 發送請求

In [ ]:
!pip install requests pandas beautifulsoup4 lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print('✅ 所有套件載入成功')

In [ ]:
# ▶ 發送一次請求，只要 2 筆資料

url = 'http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2'
response = requests.get(url)

print('HTTP 狀態碼：', response.status_code)
# 200 = 成功，429 = 請求太頻繁，500 = 伺服器錯誤

In [ ]:
# ▶ 看看回傳的原始內容（前 300 字元）

print(response.text[:300])

In [ ]:
# ▶ 印出完整 XML

print(response.text)

---
# 階段三：認識 BeautifulSoup

In [ ]:
# ▶ 用 BeautifulSoup 解析 response.text

soup    = BeautifulSoup(response.text, 'xml')
entries = soup.find_all('entry')

print('找到幾個 entry：', len(entries))

---
# 階段四：逐步取出每個欄位

In [ ]:
# ▶ 取出第一篇論文的各個欄位

entry_1 = entries[0]

title     = entry_1.find('title').text.strip()
summary   = entry_1.find('summary').text.strip()
published = entry_1.find('published').text[:10]
link      = entry_1.find('id').text

authors_list = []
for a in entry_1.find_all('author'):
    authors_list.append(a.find('name').text)

print('📌 標題  ：', title)
print('📅 日期  ：', published)
print('👤 作者  ：', authors_list)
print('🔗 連結  ：', link)
print('\n📝 摘要：')
print(summary)

---
# 階段五：用物件導向（Class）整理程式碼

## 為什麼用物件導向？

函式版本每次呼叫都是獨立的，資料用完就不見了：

```python
# 函式版本
papers = fetch_arxiv_papers(query, max_results)
# 只能拿到 papers，其他資料（網址、參數）都不見了
```

物件導向版本把**資料和操作**都包在一起，可以隨時取用：

```python
# 物件導向版本
crawler = ArxivCrawler(max_results=10)
crawler.search('image recognition')   # 搜尋
crawler.papers                        # 取得論文資料
crawler.save_csv()                    # 儲存
crawler.search('object detection')    # 再搜尋新的
```

---

## Class 的基本結構

```python
class 類別名稱:

    def __init__(self, 初始參數):
        # 物件建立時執行，用來設定初始資料
        self.資料 = 初始值

    def 方法名稱(self):
        # 物件可以執行的動作
        pass
```

- `__init__`：物件建立時自動執行，設定初始狀態
- `self`：代表「這個物件自己」，用來存取物件內部的資料
- 方法（method）：物件可以執行的動作，就像函式但屬於這個 Class

In [ ]:
class ArxivCrawler:
    """
    arXiv 論文爬蟲
    負責搜尋論文、解析資料、儲存結果
    """

    def __init__(self, max_results=10):
        """
        物件建立時執行
        設定初始資料：API 網址、最大筆數、空的論文串列

        參數：
            max_results: 每次搜尋最多回傳幾筆，預設 10
        """
        # self.xxx 代表這個物件自己擁有的資料
        self.base_url    = 'http://export.arxiv.org/api/query'  # API 網址
        self.max_results = max_results                           # 最大筆數
        self.papers      = []                                    # 存放論文的串列
        self.query       = ''                                    # 目前的搜尋關鍵字

        print(f'✅ ArxivCrawler 建立完成，每次最多爬取 {self.max_results} 筆')


    def search(self, query):
        """
        發送請求，搜尋論文

        參數：
            query: 搜尋關鍵字，例如 'cat:cs.CV AND image recognition'
        """
        self.query = query   # 把搜尋關鍵字存起來，之後可以查看

        # 設定請求參數
        params = {
            'search_query': query,
            'start'       : 0,
            'max_results' : self.max_results,
            'sortBy'      : 'submittedDate',
            'sortOrder'   : 'descending'
        }

        # 發送請求
        print(f'🔍 搜尋中：{query}')
        response = requests.get(self.base_url, params=params)

        # 確認請求成功
        if response.status_code != 200:
            print(f'❌ 請求失敗，狀態碼：{response.status_code}')
            return

        # 解析 XML
        self.papers = self._parse(response.text)
        print(f'✅ 搜尋完成，共取得 {len(self.papers)} 篇論文')


    def _parse(self, xml_text):
        """
        解析 XML，取出每篇論文的資料
        方法名稱前面加底線 _ 代表這是內部使用的方法，不需要從外部呼叫

        參數：
            xml_text: 從 API 回傳的 XML 字串（response.text）
        回傳：
            papers: 論文資料的串列
        """
        soup    = BeautifulSoup(xml_text, 'xml')
        entries = soup.find_all('entry')

        papers = []
        for entry in entries:

            # 取出作者串列
            authors_list = []
            for a in entry.find_all('author'):
                authors_list.append(a.find('name').text)

            paper = {
                '論文名'  : entry.find('title').text.strip(),
                '摘要'    : entry.find('summary').text.strip(),
                '發表時間' : entry.find('published').text[:10],
                '作者'    : authors_list,
                '論文網址' : entry.find('id').text
            }
            papers.append(paper)

        return papers


    def show(self):
        """
        把論文資料顯示成 DataFrame 表格
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        df = pd.DataFrame(self.papers)
        return df


    def show_paper(self, index=0):
        """
        顯示單篇論文的詳細資訊

        參數：
            index: 第幾篇論文，預設第一篇（index=0）
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        paper = self.papers[index]
        print(f'📌 標題  ：{paper["論文名"]}')
        print(f'📅 日期  ：{paper["發表時間"]}')
        print(f'👤 作者  ：{", ".join(paper["作者"])}')
        print(f'🔗 連結  ：{paper["論文網址"]}')
        print(f'\n📝 摘要：')
        print(paper['摘要'])


    def save_csv(self, filename='cv_papers.csv'):
        """
        把論文資料儲存成 CSV 檔案

        參數：
            filename: 檔案名稱，預設 'cv_papers.csv'
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        df = pd.DataFrame(self.papers)
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f'✅ 已儲存！共 {len(self.papers)} 筆論文 → {filename}')


print('✅ ArxivCrawler Class 定義完成')

---
## Class 結構總覽

```
ArxivCrawler
│
├── 資料（屬性）
│   ├── self.base_url     API 網址
│   ├── self.max_results  最大筆數
│   ├── self.papers       論文串列
│   └── self.query        目前搜尋關鍵字
│
└── 動作（方法）
    ├── __init__()    建立物件，設定初始資料
    ├── search()      發送請求，搜尋論文
    ├── _parse()      解析 XML（內部使用）
    ├── show()        顯示成 DataFrame 表格
    ├── show_paper()  顯示單篇論文詳細資訊
    └── save_csv()    儲存成 CSV 檔案
```

---
# 階段六：執行爬蟲，搜尋電腦視覺論文

In [ ]:
# ▶ 建立爬蟲物件
# 這裡會自動執行 __init__

crawler = ArxivCrawler(max_results=10)

In [ ]:
# ▶ 搜尋論文

crawler.search('cat:cs.CV AND (image recognition OR object detection)')

In [ ]:
# ▶ 顯示成表格

crawler.show()

In [ ]:
# ▶ 查看第一篇論文詳細資訊

crawler.show_paper(index=0)

In [ ]:
# ▶ 查看第二篇論文詳細資訊

crawler.show_paper(index=1)

In [ ]:
# ▶ 直接取得論文資料（self.papers 存在物件裡，隨時可以取用）

print('目前搜尋關鍵字：', crawler.query)
print('論文總篇數：', len(crawler.papers))
print('第一篇標題：', crawler.papers[0]['論文名'])

In [ ]:
# ▶ 換一個關鍵字搜尋（不需要重新建立物件）

time.sleep(3)  # 等 3 秒，避免請求太頻繁
crawler.search('cat:cs.CV AND image segmentation')

In [ ]:
# ▶ 儲存成 CSV

crawler.save_csv('cv_papers_oop.csv')

In [ ]:
# ▶ 多個關鍵字搜尋，合併結果

queries = [
    'cat:cs.CV AND image recognition',
    'cat:cs.CV AND object detection',
    'cat:cs.CV AND image segmentation'
]

all_papers = []

for q in queries:
    crawler.search(q)
    all_papers.extend(crawler.papers)   # 把每次結果合併
    time.sleep(3)

# 去除重複論文
df_all = pd.DataFrame(all_papers).drop_duplicates(subset='論文網址')
print(f'\n✅ 總共爬取到 {len(df_all)} 篇不重複論文')

# 儲存合併結果
df_all.to_csv('cv_papers_all.csv', index=False, encoding='utf-8-sig')
print('✅ 已儲存 → cv_papers_all.csv')